# DA6401 Report - Section 2.7 Global Performance Analysis

Overlay **training vs test accuracy** across hyperparameter search runs,
and identify high-train/low-test gap runs (overfitting-like behavior).


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import wandb

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

print('Project root:', ROOT)


In [ ]:
RUN_EXPERIMENT = False  # Set True to fetch runs and log the 2.7 summary run.

PROJECT = 'da6401_assignment_1'
ENTITY = 'da25s007-'
SWEEP_ID = 'slv9uapx'
OUT_DIR = ROOT / 'src' / 'tmp'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def collect_finished_runs(entity: str, project: str, sweep_id: str):
    api = wandb.Api()
    sweep = api.sweep(f'{entity}/{project}/{sweep_id}')
    rows = []
    for r in sweep.runs:
        if str(r.state).lower() != 'finished':
            continue
        s = r.summary._json_dict
        train_acc = s.get('train/accuracy')
        test_acc = s.get('test/accuracy')
        val_acc = s.get('val/accuracy')
        if train_acc is None or test_acc is None:
            continue
        rows.append(
            {
                'run_id': r.id,
                'run_name': r.name,
                'train_acc': float(train_acc),
                'val_acc': float(val_acc) if val_acc is not None else np.nan,
                'test_acc': float(test_acc),
            }
        )
    for row in rows:
        row['gap'] = row['train_acc'] - row['test_acc']
    rows.sort(key=lambda x: x['gap'], reverse=True)
    return rows


def make_plots(rows):
    idx = np.arange(len(rows))
    train = np.array([r['train_acc'] for r in rows], dtype=float)
    test = np.array([r['test_acc'] for r in rows], dtype=float)

    fig1, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(idx, train, label='Train accuracy', linewidth=1.8)
    ax1.plot(idx, test, label='Test accuracy', linewidth=1.8)
    ax1.set_title('2.7: Training vs Test accuracy across all sweep runs')
    ax1.set_xlabel('Run index (sorted by train-test gap descending)')
    ax1.set_ylabel('Accuracy')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    overlay_path = OUT_DIR / 'report_2_7_train_vs_test_overlay.png'
    fig1.savefig(overlay_path, dpi=160, bbox_inches='tight')
    plt.close(fig1)

    fig2, ax2 = plt.subplots(figsize=(7, 6))
    ax2.scatter(train, test, alpha=0.75)
    lo = float(min(train.min(), test.min()))
    hi = float(max(train.max(), test.max()))
    ax2.plot([lo, hi], [lo, hi], linestyle='--', linewidth=1.2, label='y = x')
    ax2.set_title('2.7: Train vs Test scatter (all sweep runs)')
    ax2.set_xlabel('Train accuracy')
    ax2.set_ylabel('Test accuracy')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    scatter_path = OUT_DIR / 'report_2_7_train_vs_test_scatter.png'
    fig2.savefig(scatter_path, dpi=160, bbox_inches='tight')
    plt.close(fig2)

    return overlay_path, scatter_path


if RUN_EXPERIMENT:
    rows = collect_finished_runs(ENTITY, PROJECT, SWEEP_ID)
    overlay_path, scatter_path = make_plots(rows)

    run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name='report_2_7_summary',
        tags=['report', '2.7', 'global_performance', 'summary'],
        config={'source_sweep_id': SWEEP_ID, 'num_runs': len(rows)},
    )

    table_all = wandb.Table(columns=['run_id', 'run_name', 'train_acc', 'val_acc', 'test_acc', 'gap'])
    for r in rows:
        table_all.add_data(r['run_id'], r['run_name'], r['train_acc'], r['val_acc'], r['test_acc'], r['gap'])

    top_k = rows[:10]
    table_gap = wandb.Table(columns=['run_id', 'run_name', 'train_acc', 'test_acc', 'gap'])
    for r in top_k:
        table_gap.add_data(r['run_id'], r['run_name'], r['train_acc'], r['test_acc'], r['gap'])

    run.log(
        {
            'analysis/train_test_overlay_plot': wandb.Image(str(overlay_path)),
            'analysis/train_test_scatter_plot': wandb.Image(str(scatter_path)),
            'analysis/global_performance_table': table_all,
            'analysis/overfit_like_runs_table': table_gap,
        }
    )

    gaps = np.array([r['gap'] for r in rows], dtype=float)
    run.summary['stats/num_runs'] = int(len(rows))
    run.summary['stats/mean_gap'] = float(np.mean(gaps))
    run.summary['stats/max_gap'] = float(np.max(gaps))
    run.summary['stats/min_gap'] = float(np.min(gaps))
    run.finish()

    payload = {
        'summary_run_name': 'report_2_7_summary',
        'source_sweep_id': SWEEP_ID,
        'num_runs': len(rows),
        'top_gap_runs': rows[:10],
        'artifacts': {
            'overlay_plot': str(overlay_path),
            'scatter_plot': str(scatter_path),
        },
    }
    out_path = OUT_DIR / 'report_2_7_global_performance.json'
    out_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('Wrote', out_path)
else:
    print('RUN_EXPERIMENT=False -> no W&B calls made.')


In [ ]:
summary_path = ROOT / 'src' / 'tmp' / 'report_2_7_global_performance.json'
if summary_path.exists():
    data = json.loads(summary_path.read_text())
    print('Summary file:', summary_path)
    print('Source sweep:', data.get('source_sweep_id'))
    print('Num runs:', data.get('num_runs'))
    for row in data.get('top_gap_runs', [])[:5]:
        print(
            row['run_id'],
            row['run_name'],
            'train=', round(row['train_acc'], 4),
            'test=', round(row['test_acc'], 4),
            'gap=', round(row['gap'], 4),
        )
else:
    print('No summary JSON yet. Set RUN_EXPERIMENT=True and run the previous cell.')
